# Meteoritos: de un fichero CSV a un proyecto de análisis
### Computación Científica I

| Parte | Qué haces | Cómo se trabaja |
|---|---|---|
| **Parte 1. Demo guiada con retos** | Recorremos las funcionalidades del proyecto (lectura, limpieza, estadísticas, filtros, informe y gráficos) con los datos reales. Tras cada bloque hay un **reto** corto. | En clase, individual o por parejas |
| **Parte 2. Vuestro proyecto** | Convertís lo aprendido en `proyecto.py`, tomáis decisiones de diseño como equipo y elegís una **ruta de exploración**. | En equipo (3 o 4 personas) |

Herramientas: `pandas` para los datos y `matplotlib` para los gráficos. En la Parte 2 aparecen además `reverse_geocoder` y `pycountry_convert`.

# Parte 1. Demo guiada con retos

## 0. Los datos: de dónde vienen

Usamos el conjunto **Meteorite Landings**, publicado en el **Portal de Datos Abiertos de la NASA**. Los datos los recopila **The Meteoritical Society**, la asociación científica internacional que mantiene el registro oficial de meteoritos (el *Meteoritical Bulletin*). Contiene unos 45.000 meteoritos conocidos, tanto los que se vieron caer como los que se encontraron después.

**Antes de seguir, abre la página de documentación del conjunto de datos:**

**https://data.nasa.gov/dataset/meteorite-landings**

Fíjate en:
* qué organización produce los datos y cuándo se actualizaron por última vez;
* qué formatos de descarga se ofrecen (CSV, JSON, XML...);
* qué columnas tiene el fichero y qué significa cada una (compáralas con la tabla del enunciado).

Leer la documentación **antes** de programar es una buena práctica: te dice, por ejemplo, en qué unidades está la masa. Lo que la documentación *no* dice (valores ausentes, coordenadas raras, años imposibles) lo descubrirás explorando, como haremos en los retos.

`pd.read_csv` puede leer directamente desde una URL: no hace falta descargar nada a mano.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL_DATOS = "https://data.nasa.gov/docs/legacy/meteorite_landings/Meteorite_Landings.csv"
meteoritos = pd.read_csv(URL_DATOS)

meteoritos.head()

Las primeras preguntas ante cualquier tabla nueva: **¿cuántas filas y columnas tiene?** y **¿de qué tipo es cada columna?**

In [ ]:
print("Filas y columnas:", meteoritos.shape)
meteoritos.info()

Compara los tipos con la tabla del enunciado:

| Columna | Lo que esperamos | Lo que ha leído pandas |
|---|---|---|
| `mass (g)`, `reclat`, `reclong` | números | `float64`: bien |
| `year` | un año (entero) | `object` (texto): **hay que arreglarlo** |
| `GeoLocation` | coordenadas | texto con `(lat, lon)`: es redundante con `reclat` y `reclong` |

El campo `info()` también muestra cuántos valores **no nulos** tiene cada columna: si es menor que el número de filas, faltan datos.

### Reto 1. ¿Qué datos faltan?
1. Cuenta los valores ausentes de cada columna (pista: `.isna().sum()`).
2. Cuenta cuántos meteoritos tienen coordenadas **exactamente (0, 0)** y cuántos tienen **masa 0**.
3. *Para pensar:* ¿crees que hay tantos meteoritos caídos justo en el punto (0, 0) del Atlántico? ¿Qué significa realmente ese valor?

In [ ]:
# TU CÓDIGO AQUÍ

## 1. Limpieza: decisiones explícitas

Limpiar no es "borrar lo que molesta": es **tomar decisiones y dejarlas escritas**. Creamos columnas nuevas y limpias (`anio`, `lat`, `lon`) y conservamos las originales, para poder comparar.

**El año.** En el fichero de la NASA el año viene como texto, por ejemplo `01/01/1880 12:00:00 AM`. Podríamos usar `pd.to_datetime`, pero pandas no representa fechas anteriores a 1677 y aquí hay meteoritos del siglo IX. Así que extraemos los 4 dígitos del año con una expresión regular. (Si tu fichero ya trae el año como número, la misma línea funciona.)

In [ ]:
meteoritos["anio"] = pd.to_numeric(
    meteoritos["year"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce"
).astype("Int64")

print("Año mínimo:", meteoritos["anio"].min(), "  Año máximo:", meteoritos["anio"].max())
meteoritos[["name", "year", "anio"]].head()

**Decisiones de la demo** (en la Parte 2 tu equipo puede decidir otra cosa, pero debe justificarlo):
1. **Coordenadas (0, 0)**: las tratamos como desconocidas (`NaN`).
2. **Años posteriores al año actual**: imposibles, los marcamos como desconocidos.
3. **Masa**: la dejamos como está; una masa 0 es sospechosa, pero no la tocamos todavía.

`NaN` ("not a number") significa "no lo sabemos". Es mejor que un 0, porque un 0 entra en las medias y en los mapas como si fuera un dato real.

In [ ]:
ANIO_ACTUAL = 2026

coord_cero = (meteoritos["reclat"] == 0) & (meteoritos["reclong"] == 0)
meteoritos["lat"] = meteoritos["reclat"].mask(coord_cero)
meteoritos["lon"] = meteoritos["reclong"].mask(coord_cero)
meteoritos["anio"] = meteoritos["anio"].mask(meteoritos["anio"] > ANIO_ACTUAL)

print("Coordenadas (0, 0) convertidas en NaN:", coord_cero.sum())
print("Meteoritos con coordenadas válidas:", meteoritos["lat"].notna().sum())

## 2. Estadísticas básicas

### 2a, 2b y 2c. Total, más pesado, más ligero y media

* `len(df)` da el número de filas.
* `idxmax()` / `idxmin()` devuelven la **etiqueta de la fila** donde está el máximo / mínimo; con `.loc[...]` recuperamos la fila completa.
* `mean()` ignora los `NaN` automáticamente.

In [ ]:
masa = meteoritos["mass (g)"]
mas_pesado = meteoritos.loc[masa.idxmax()]
mas_ligero = meteoritos.loc[masa.idxmin()]

print(f"Número total de meteoritos: {len(meteoritos)}")
print(f"Más pesado: {mas_pesado['name']} ({mas_pesado['mass (g)']:,.0f} g)")
print(f"Más ligero: {mas_ligero['name']} ({mas_ligero['mass (g)']} g)")
print(f"Media de masas: {masa.mean():,.1f} g  (sobre {masa.notna().sum()} meteoritos con masa)")

In [ ]:
# describe() resume todo de una vez
masa.describe()

### Reto 2. ¿Es la media un buen resumen?
1. Calcula la **mediana** de las masas y compárala con la media. ¿Por qué son tan diferentes? (Mira también el resultado de `describe()`.)
2. Muestra los **10 meteoritos más pesados** con su nombre, masa en kg y año (pista: `nlargest`).
3. ¿El más ligero tiene sentido? ¿Cuántos meteoritos pesan menos de 1 g?

In [ ]:
# TU CÓDIGO AQUÍ

### 2d. Meteoritos por franjas temporales, por estado y combinados

* **Franja**: el año de inicio de la franja es `(anio // ancho) * ancho`. El ancho es una **decisión de diseño**, así que lo dejamos como parámetro.
* **Conteos simples**: `value_counts()`.
* **Combinado**: `pd.crosstab(filas, columnas)` cuenta cada combinación y devuelve una tabla.

In [ ]:
ANCHO_FRANJA = 50

meteoritos["franja"] = (meteoritos["anio"] // ANCHO_FRANJA) * ANCHO_FRANJA

print("Por estado:")
print(meteoritos["fall"].value_counts(), "\n")

tabla = pd.crosstab(meteoritos["franja"], meteoritos["fall"], margins=True, margins_name="Total")
tabla.index = [f"{int(i)}-{int(i) + ANCHO_FRANJA - 1}" if i != "Total" else i for i in tabla.index]
tabla

### Reto 3. Elige tus franjas
1. Cambia `ANCHO_FRANJA` a 100 y a 25 y vuelve a ejecutar la celda anterior. ¿Qué ancho cuenta mejor la historia? ¿Hay franjas con tan pocos datos que no dicen nada?
2. Calcula, para cada franja, el **porcentaje de meteoritos observados al caer (`Fell`)** (pista: `pd.crosstab(..., normalize="index")`). ¿Cuándo empieza a dominar `Found`? ¿Se te ocurre por qué? (Busca "Antarctic Search for Meteorites".)

In [ ]:
# TU CÓDIGO AQUÍ

### 2e. Porcentaje de meteoritos por continente

El enunciado define cada continente como una **caja** de latitudes y longitudes. Antes de programar, **mira las cajas en un mapa**: vas a descubrir dos problemas que tu equipo tendrá que resolver.

In [ ]:
from matplotlib.patches import Rectangle

CONTINENTES = {                       # (lat_min, lat_max, lon_min, lon_max)
    "América del Norte": (7.0, 83.0, -172.0, -10.0),
    "América del Sur":   (-55.0, 12.0, -81.0, -35.0),
    "Europa":            (35.0, 71.0, -25.0, 60.0),
    "África":            (-35.0, 37.0, -18.0, 52.0),
    "Asia":              (-1.0, 81.0, 26.0, 180.0),
    "Oceanía":           (-47.0, 10.0, 110.0, 180.0),
}

COLORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
COLOR_FELL, COLOR_FOUND = "#2a78d6", "#eb6834"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.edgecolor": "#52514e", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "font.size": 10})

con_coord = meteoritos.dropna(subset=["lat", "lon"])

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.scatter(con_coord["lon"], con_coord["lat"], s=1, color="#52514e", alpha=0.3)
for (nombre, (la0, la1, lo0, lo1)), color in zip(CONTINENTES.items(), COLORES):
    ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, alpha=0.12, color=color))
    ax.add_patch(Rectangle((lo0, la0), lo1 - lo0, la1 - la0, fill=False, lw=2, edgecolor=color))
    ax.text(lo0 + 2, la1 - 6, nombre, fontsize=9, weight="bold")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title("Meteoritos con coordenadas y cajas de continentes del enunciado", loc="left")
plt.show()

**Los dos problemas que se ven en el mapa:**

1. **Las cajas se solapan.** El sur de España está a la vez en Europa y en África; Turquía en Europa y Asia; el norte de Sudamérica en las dos Américas. Hace falta una **regla de desempate**.
2. **Hay zonas sin caja.** Los océanos y, sobre todo, la **Antártida**, que no aparece en la tabla. Allí se ha encontrado una enorme cantidad de meteoritos: el hielo los concentra y sobre la nieve se ven fácilmente.

La demo usa la regla más sencilla: **el primer continente del diccionario que contiene el punto**. `np.select` hace exactamente eso: recibe una lista de condiciones y una lista de valores, y para cada fila elige el valor de la **primera** condición que se cumple.

In [ ]:
def en_caja(df, caja):
    """Serie booleana: True si (lat, lon) de cada fila está dentro de la caja."""
    lat_min, lat_max, lon_min, lon_max = caja
    return df["lat"].between(lat_min, lat_max) & df["lon"].between(lon_min, lon_max)

condiciones = [en_caja(meteoritos, caja) for caja in CONTINENTES.values()]
meteoritos["continente"] = np.select(condiciones, list(CONTINENTES.keys()), default="Sin región")
meteoritos.loc[meteoritos["lat"].isna(), "continente"] = "Sin coordenadas"

resumen = meteoritos["continente"].value_counts().to_frame("meteoritos")
resumen["%"] = (100 * resumen["meteoritos"] / len(meteoritos)).round(1)
resumen

### Reto 4. ¿Cuánto importan los solapes?
1. Para cada meteorito con coordenadas, cuenta **en cuántas cajas** cae (pista: suma las Series booleanas de `condiciones`). ¿Cuántos caen en 0, 1 o 2 cajas?
2. Cuenta cuántos meteoritos tienen latitud menor que -60 (Antártida). ¿Qué porcentaje del total representan?
3. Debate: ¿el porcentaje por continente debería calcularse sobre **todos** los meteoritos o solo sobre los que tienen coordenadas? Escribe tu decisión y tu argumento en una celda Markdown.

In [ ]:
# TU CÓDIGO AQUÍ

## 3. Filtrado y búsqueda

En pandas se filtra con **máscaras booleanas**: una Serie de `True`/`False` del mismo largo que la tabla. `df[mascara]` se queda con las filas `True`. Las máscaras se combinan con `&` (y), `|` (o) y `~` (no), siempre entre paréntesis.

In [ ]:
COLUMNAS_VISTA = ["name", "anio", "mass (g)", "fall", "recclass", "continente"]

# Un año concreto
print("Meteoritos de 1950:")
print(meteoritos[meteoritos["anio"] == 1950][COLUMNAS_VISTA].head(), "\n")

# Un rango de años
siglo_xx = meteoritos[meteoritos["anio"].between(1900, 1999)]
print("Entre 1900 y 1999:", len(siglo_xx), "\n")

# Por nombre, sin distinguir mayúsculas
print("Nombre contiene 'madrid':")
print(meteoritos[meteoritos["name"].str.contains("madrid", case=False)][COLUMNAS_VISTA])

### Reto 5. Combinar filtros
1. ¿Cuántos meteoritos se **vieron caer** (`Fell`) entre 1900 y 1999? ¿Cuál fue el más pesado?
2. ¿Cuántos de ellos están en Europa?
3. Escribe una función `buscar(df, desde=None, hasta=None, nombre=None, estado=None)` que aplique solo los filtros que reciba.

In [ ]:
# TU CÓDIGO AQUÍ

## 4. Salida en fichero: el informe

Una función construye el texto del informe a partir de **cualquier** tabla (la completa o el resultado de un filtro) y otra lo escribe en disco. `to_string()` convierte una tabla de pandas en texto alineado.

In [ ]:
def texto_informe(df, titulo):
    """Devuelve el informe de una tabla de meteoritos como un único texto."""
    m = df["mass (g)"]
    partes = [titulo, "=" * len(titulo), "",
              f"Número de meteoritos: {len(df)}"]
    if m.notna().any():
        partes += [f"Más pesado: {df.loc[m.idxmax(), 'name']} ({m.max():,.0f} g)",
                   f"Más ligero: {df.loc[m.idxmin(), 'name']} ({m.min()} g)",
                   f"Media de masas: {m.mean():,.1f} g",
                   f"Mediana de masas: {m.median():,.1f} g"]
    partes += ["", "Por estado:", df["fall"].value_counts().to_string(), "",
               "Por continente (%):",
               (100 * df["continente"].value_counts(normalize=True)).round(1).to_string()]
    return "\n".join(partes)


def escribir_informe(ruta, texto):
    with open(ruta, "w", encoding="utf-8") as f:
        f.write(texto)
    print("Informe guardado en", ruta)


informe = texto_informe(meteoritos, "Informe general de meteoritos")
escribir_informe("informe_general.txt", informe)
print(informe)

### Reto 6. Informe de una consulta
Genera `informe_siglo_XX.txt` con el informe de los meteoritos de **1900 a 1999**, y añade una sección con el número de meteoritos **por década** de ese siglo. Guarda también la tabla filtrada como `siglo_XX.csv` (pista: `to_csv`).

In [ ]:
# TU CÓDIGO AQUÍ

## 5. Gráficos con matplotlib

Tres gráficos, cada uno elegido por la **pregunta** que responde:

| Pregunta | Gráfico |
|---|---|
| ¿Cómo se reparten las masas? | Histograma del **logaritmo** de la masa (van de gramos a toneladas: en escala lineal todo se aplasta en la primera barra) |
| ¿Cómo cambia Fell/Found con el tiempo? | Barras agrupadas por franja |
| ¿Dónde caen? | Dispersión latitud/longitud coloreada por estado |

In [ ]:
log_masa = np.log10(masa[masa > 0])

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(log_masa, bins=40, color=COLOR_FELL, edgecolor="white", linewidth=1)
ax.set_xticks(range(-1, 9))
ax.set_xticklabels(["0,1 g", "1 g", "10 g", "100 g", "1 kg", "10 kg", "100 kg", "1 t", "10 t", "100 t"])
ax.set_xlabel("Masa (escala logarítmica)")
ax.set_ylabel("Número de meteoritos")
ax.set_title("Distribución de masas", loc="left")
plt.show()

In [ ]:
por_franja = pd.crosstab(meteoritos["franja"], meteoritos["fall"])
por_franja = por_franja[por_franja.index >= 1700]          # las franjas antiguas tienen muy pocos datos
por_franja.index = [f"{int(i)}-{int(i) + ANCHO_FRANJA - 1}" for i in por_franja.index]

ax = por_franja.plot(kind="bar", figsize=(10, 4), color=[COLOR_FELL, COLOR_FOUND], width=0.8)
ax.set_xlabel("")
ax.set_ylabel("Número de meteoritos")
ax.set_title(f"Meteoritos por franja de {ANCHO_FRANJA} años y estado", loc="left")
ax.legend(["Fell (observado al caer)", "Found (hallado)"], frameon=False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.5))
for estado, color, tam in [("Found", COLOR_FOUND, 1), ("Fell", COLOR_FELL, 6)]:
    sub = con_coord[con_coord["fall"] == estado]
    ax.scatter(sub["lon"], sub["lat"], s=tam, color=color, alpha=0.6, label=f"{estado} ({len(sub)})")
ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
ax.set_xlabel("Longitud"); ax.set_ylabel("Latitud")
ax.set_title("¿Dónde están los meteoritos? Observados frente a hallados", loc="left")
ax.legend(frameon=False, loc="lower left", markerscale=4)
plt.show()

### Reto 7. Tu gráfico
Elige **una** pregunta y respóndela con un gráfico (título, ejes con nombre y leyenda si hay más de una serie):

* ¿Qué porcentaje de meteoritos hay en cada continente? (barras horizontales ordenadas)
* ¿Las masas de los observados (`Fell`) son distintas de las de los hallados (`Found`)? (dos histogramas del logaritmo de la masa)
* ¿Cuáles son las 10 clases (`recclass`) más frecuentes?

Escribe debajo, en una celda Markdown, **una frase** con la conclusión que se ve en tu gráfico.

In [ ]:
# TU CÓDIGO AQUÍ

# Parte 2. Vuestro proyecto

En la Parte 1 has visto **una** forma de resolver cada funcionalidad. Ahora vuestro equipo construye `proyecto.py` y **toma decisiones propias**. No hay una única respuesta correcta: se valora que cada decisión esté **justificada** y documentada en la memoria y en la presentación.

## 2.1 Del cuaderno al programa: diseño arquitectónico

El enunciado pide que **cada función sea un módulo** y que se represente el **intercambio de datos con el programa principal**:

```
                          ┌──────────────────────────┐
                          │     programa principal   │
                          │  (menú en bucle: input)  │
                          └────────────┬─────────────┘
           URL o ruta (str)            │
   ┌───────────────────────────────────┼─────────────────────────────────────┐
   ▼                                   ▼                                     ▼
cargar_datos -> DataFrame       limpiar(df) -> DataFrame              filtrar(df, ...) -> DataFrame
                                       │                                     │
                                       ▼                                     ▼
                          estadisticas(df) -> dict / Series     texto_informe(df) -> str -> escribir_informe
                                       │
                                       ▼
                               graficar_*(df) -> figura / .png
```

Regla de oro: **los datos se cargan una vez** en el programa principal y se pasan como argumento. Las funciones **devuelven** resultados en lugar de imprimirlos; así sirven para la pantalla, el informe y los gráficos.

### Esqueleto de `proyecto.py`

```python
# proyecto.py  -  Computación Científica I  -  Equipo: ____________
# Autores: ______, ______, ______, ______

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL_DATOS = "https://data.nasa.gov/docs/legacy/meteorite_landings/Meteorite_Landings.csv"
CONTINENTES = { ... }          # tabla del enunciado


def cargar_datos(origen):
    '''Entradas: origen (str, URL o ruta).  Salidas: DataFrame o None.
    Proceso: pd.read_csv con gestión de errores (fichero no encontrado, sin conexión, formato).'''
    pass


def limpiar(df):
    '''Entradas: DataFrame original.  Salidas: DataFrame con columnas anio, lat, lon, franja, continente.
    Proceso: aplica las decisiones de limpieza del equipo.'''
    pass


def estadisticas(df):
    pass


def filtrar(df, desde=None, hasta=None, nombre=None):
    pass


def texto_informe(df, titulo):
    pass


def escribir_informe(ruta, texto):
    pass


def menu():
    print("1. Estadísticas generales")
    print("2. Filtrar por año o rango de años")
    print("3. Buscar por nombre")
    print("4. Guardar informe de la última consulta")
    print("0. Salir")
    return input("Opción: ")


def main():
    datos = cargar_datos(URL_DATOS)
    if datos is None:
        return
    datos = limpiar(datos)
    seleccion = datos            # resultado de la última consulta
    opcion = menu()
    while opcion != "0":
        ...
        opcion = menu()


if __name__ == "__main__":
    main()
```

El menú con `input()` está pensado para ejecutarse en la terminal (`python proyecto.py`), no en el cuaderno.

**Gestión de errores en la carga.** `pd.read_csv` puede fallar por varios motivos; capturadlos para que el programa no se rompa:

In [ ]:
def cargar_datos(origen):
    """Lee el CSV de meteoritos desde una URL o un fichero local.

    Entradas: origen (str), URL o ruta del fichero.
    Salidas:  DataFrame con los datos, o None si no se pudo leer.
    Proceso:  pd.read_csv dentro de try/except; comprueba que estén las columnas necesarias.
    """
    necesarias = {"name", "id", "nametype", "recclass", "mass (g)", "fall", "year", "reclat", "reclong"}
    try:
        df = pd.read_csv(origen)
    except FileNotFoundError:
        print(f"No se encuentra el fichero '{origen}'.")
        return None
    except OSError as error:                 # sin conexión, URL incorrecta...
        print("No se pudo acceder a los datos:", error)
        return None
    except pd.errors.ParserError:
        print("El fichero no tiene formato CSV válido.")
        return None
    faltan = necesarias - set(df.columns)
    if faltan:
        print("Formato incorrecto. Faltan las columnas:", sorted(faltan))
        return None
    return df


print(cargar_datos("no_existe.csv"))

## 2.2 Decisiones de diseño del equipo

Rellenad esta tabla **juntos** en la primera reunión y apuntad la decisión en el fichero de seguimiento. Cada fila es material directo para la sección "Descripción de los datos" de la memoria y para la presentación.

| # | Decisión | Opciones posibles | Nuestra decisión | Por qué |
|---|---|---|---|---|
| 1 | Origen de los datos | leer siempre de la URL / descargar una vez y leer el fichero local | | |
| 2 | Datos ausentes | `NaN` y se ignoran / se eliminan filas (`dropna`) / se rellenan | | |
| 3 | Coordenadas (0, 0) | dato válido / desconocido | | |
| 4 | Ancho de las franjas temporales | 10 / 25 / 50 / 100 años / franjas irregulares con `pd.cut` | | |
| 5 | Años imposibles o muy antiguos | se descartan / se agrupan en "antes de 1800" | | |
| 6 | Solape de cajas de continentes | primero en el orden / caja más pequeña / se cuenta en todos / categoría "frontera" | | |
| 7 | Puntos sin caja (océano, Antártida) | "Sin región" / añadir Antártida y océanos como regiones propias | | |
| 8 | Base de los porcentajes | todos los meteoritos / solo los que tienen coordenadas | | |
| 9 | Búsqueda por nombre | exacta / contiene / sin distinguir mayúsculas ni tildes | | |
| 10 | Formato del informe | texto plano / tabla alineada / CSV | | |
| 11 | Ruta de exploración | A, B, C, D o E (ver 2.3) | | |

Si vuestro código lee estas decisiones de **constantes**, podéis cambiarlas sin tocar las funciones (y enseñar en la presentación cómo cambia el resultado):

In [ ]:
# ---- Configuración del equipo (editad estos valores) ----
ANCHO_FRANJA = 50               # decisión 4
CERO_ES_DESCONOCIDO = True      # decisión 3
REGLA_SOLAPE = "primero"        # decisión 6: "primero", "mas_pequena", "todos"
ANIO_MAXIMO_VALIDO = 2026       # decisión 5

## 2.3 Elegid una ruta de exploración

Cada ruta parte de una **pregunta**. Para la ruta elegida, el entregable mínimo es: **una pregunta, al menos dos gráficos, una tabla en el informe y una conclusión de un párrafo** que responda a la pregunta.

### Ruta A. ¿Quién mira al cielo? Observados frente a hallados
*Pregunta guía:* ¿El número de meteoritos registrados dice algo sobre el cielo o sobre las personas que los buscan?
* Proporción `Fell`/`Found` por franja temporal y por continente.
* ¿Desde qué década domina `Found`? Relacionadlo con las campañas de búsqueda en la Antártida y en desiertos (Sáhara, Omán).
* ¿Dónde se *observan* caer más meteoritos? ¿Coincide con zonas muy pobladas?
* Gráficos sugeridos: barras apiladas en porcentaje por franja; mapa de solo `Fell`.

### Ruta B. Geografía de verdad: cajas frente a fronteras
*Pregunta guía:* ¿Cuánto se equivoca el método de las cajas?
* Usad `reverse_geocoder` y `pycountry_convert` (celda de ejemplo más abajo) para asignar **país y continente reales**.
* Construid una **tabla de confusión** con `pd.crosstab(continente_cajas, continente_real)`. ¿Dónde están los errores?
* Cuidado: `reverse_geocoder` devuelve la **ciudad más cercana**, aunque el punto esté en mitad del océano o en la Antártida. ¿Cómo lo detectáis?

### Ruta C. ¿De qué están hechos?
*Pregunta guía:* ¿Qué tipos de meteoritos son más frecuentes y cuáles son más pesados?
* `recclass` tiene cientos de valores. Diseñad una función que la agrupe en **familias** (condritas ordinarias H, L y LL; condritas carbonáceas C...; hierros `Iron...`; acondritas; pallasitas). La regla es vuestra decisión.
* Masa media y mediana por familia con `groupby`. ¿Los hierros son más pesados? ¿Por qué tendría sentido?
* ¿Hay familias que se *encuentran* mucho más de lo que se *observan* caer?

### Ruta D. Más cerca de casa: Europa y España
*Pregunta guía:* ¿Qué país europeo tiene más meteoritos y qué provincias españolas?
* Con `reverse_geocoder`, el campo `cc` da el país y `admin1`/`admin2` la comunidad y la provincia.
* Ranking de países europeos; meteoritos en España por comunidad o provincia.
* Ficha de los meteoritos españoles (nombre, año, masa, estado). ¿Alguno cayó cerca de vuestra ciudad?

### Ruta E. Ruta libre
Proponed vuestra propia pregunta. Debe combinar al menos dos columnas del fichero y responderse con gráficos y tablas. Validadla con el profesorado antes de empezar.

### Ejemplo de uso de `reverse_geocoder` y `pycountry_convert` (rutas B y D)

Instalación (una vez, en la terminal): `pip install reverse_geocoder pycountry_convert`

* Pasad **todas las coordenadas a la vez** en una lista: una llamada por meteorito es muy lenta.
* En cuadernos y en Windows usad `mode=1` (un solo proceso) para evitar errores de multiproceso.

In [ ]:
try:
    import reverse_geocoder as rg
    import pycountry_convert as pc
    LIBRERIAS_GEO = True
except ImportError:
    LIBRERIAS_GEO = False
    print("Faltan librerías. Instálalas con: pip install reverse_geocoder pycountry_convert")

NOMBRES_CONTINENTE = {"EU": "Europa", "AF": "África", "AS": "Asia", "NA": "América del Norte",
                      "SA": "América del Sur", "OC": "Oceanía", "AN": "Antártida"}

def continente_real(codigo_pais):
    """Convierte un código de país ISO alfa-2 en nombre de continente (o 'Desconocido')."""
    if codigo_pais == "AQ":
        return "Antártida"
    try:
        return NOMBRES_CONTINENTE[pc.country_alpha2_to_continent_code(codigo_pais)]
    except KeyError:
        return "Desconocido"


if LIBRERIAS_GEO:
    print(rg.search([(40.4168, -3.7038)], mode=1))       # Madrid

    geo = meteoritos.dropna(subset=["lat", "lon"]).copy()
    lugares = pd.DataFrame(rg.search(list(zip(geo["lat"], geo["lon"])), mode=1), index=geo.index)
    geo["pais"] = lugares["cc"]
    geo["region"] = lugares["admin1"]
    geo["continente_real"] = geo["pais"].apply(continente_real)

    print(geo[["name", "lat", "lon", "pais", "region", "continente_real"]].head())

**Para pensar antes de fiaros del resultado:** un meteorito encontrado en la Antártida, ¿qué "ciudad más cercana" os devuelve `reverse_geocoder`? Comparad `continente_real` con `continente` (cajas) para los puntos con latitud menor que -60.

## 2.4 Trabajo en equipo

### Plantilla de acta para el fichero de seguimiento
Copiad esta plantilla al inicio de **cada** reunión (un par de párrafos, nada extenso):

```
## Reunión n.º __  -  Fecha: __/__/____
Asistentes: ...
Objetivos de hoy: ...
Tareas de la reunión anterior: [hecha / en curso / pendiente] tarea - responsable
Desarrollo: (2-4 frases: qué se hizo, qué problemas surgieron, qué se decidió)
Próxima reunión: tareas y responsables
  - ... -> ...
```

### Reparto sugerido de roles
| Rol | Responsabilidad |
|---|---|
| Coordinación | Convoca reuniones, mantiene el acta al día |
| Datos | Carga, limpieza, tipos y casos raros |
| Estadísticas e informe | Funciones de cálculo y salida en fichero |
| Exploración y gráficos | Ruta elegida y matplotlib |

Todas las personas deben poder explicar **todo** el código: habrá un examen de autoría y todos participan en la presentación.

## 2.5 Lista de comprobación antes de entregar

**Código (`proyecto.py`)**
- [ ] Se ejecuta de principio a fin con `python proyecto.py` sin errores.
- [ ] Gestiona fichero no encontrado, falta de conexión y formato incorrecto.
- [ ] Cada función tiene docstring con **entradas, salidas y proceso**.
- [ ] Las funciones reciben los datos como argumento y devuelven resultados.
- [ ] Nombres descriptivos y comentarios útiles.

**Funcionalidad obligatoria**
- [ ] Total, más pesado, más ligero y media de masas.
- [ ] Conteos por franja, por estado y combinados.
- [ ] Porcentaje por continente, con la regla de solapes documentada.
- [ ] Filtros por año, rango de años y nombre.
- [ ] Informe en fichero de texto.

**Documentación**
- [ ] Memoria: introducción, diseño arquitectónico, datos, detalles del código, manual de usuario, conclusiones, uso de IA.
- [ ] Fichero de seguimiento con actas de cada sesión hasta la presentación.
- [ ] Presentación: problema, arquitectura con intercambio de datos, datos y tipos elegidos, algoritmos, código de interés, reparto del trabajo.